# 01 - Train and monitor

Trains in the background with **Pause / Resume / Stop** buttons and live curves, then evaluates
the held-out TEST block against the baselines. The notebook stays responsive during training (the
buttons only work while no cell is running). Stop ends training after the current batch; the run is
still evaluated, calibrated and saved. Every number here links to a run directory under `runs/`.

In [ ]:
# Parameters
CONFIG_PATH = "../configs/default.yaml"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
RUNS_DIR = "../runs"
OVERRIDES = {}            # e.g. {"EPOCHS": 5, "LR": 5e-4, "FOLD_INDEX": -2, "BATCH_SIZE": 64}
EPOCHS = None             # None -> Config.EPOCHS
CALIBRATE_LOSS_WEIGHTS = True

In [ ]:
from IPython.display import Markdown, display

import neural_trade  # first: on Windows it puts the CUDA DLLs on PATH before TensorFlow loads
from neural_trade.core.config import Config
from neural_trade.data.processor import split_arrays
from neural_trade.evaluation.baselines import BaselineSet
from neural_trade.evaluation.frame import PredictionFrame
from neural_trade.evaluation.report import evaluate
from neural_trade.experiments.run_context import RunContext
from neural_trade.notebook import TrainingSession
from neural_trade.registries.visualizations import Visualizations
from neural_trade.telemetry.epoch_logger import read_metrics
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU") or "none - training will run on the CPU")
cfg = Config.from_yaml(CONFIG_PATH).override(CSV_PATH=CSV_PATH, **OVERRIDES)
ctx = RunContext.create(cfg, root=RUNS_DIR, tags=["notebook"])
print("run:", ctx.run_dir)

## Train

This cell returns immediately; the dashboard below keeps updating. The next cell waits for the run to finish.

In [ ]:
session = TrainingSession(ctx.config, run_context=ctx, epochs=EPOCHS, calibrate=CALIBRATE_LOSS_WEIGHTS)
display(session.widget())
session.start()

In [ ]:
result = session.wait()   # blocks until training, evaluation and calibration are done
print(session.status, "-", len(session.history), "epochs")

## Evaluate on the TEST block

Baselines are fitted on the train block; the confidence threshold and calibration come from the cal block.

In [ ]:
blocks = split_arrays(ctx.config)
baselines = BaselineSet.fit(blocks["train"]["X"], blocks["train"]["y"], blocks["train"]["last_close"],
                            ctx.config.DIR_DEADBAND_BPS)
test = PredictionFrame.from_result(result, "test")
cal = PredictionFrame.from_result(result, "cal")
report = evaluate(test, ctx.config, baselines=baselines, cal_frame=cal, run_id=ctx.run_id)
report.to_json(ctx.path("eval_report_test.json"))
display(Markdown(report.to_markdown(ctx.path("eval_report_test.md"))))

In [ ]:
Visualizations.build("eval_report", test, ctx.config).show()

## Learned indicator periods

In [ ]:
Visualizations.build("indicator_evolution", ctx.path("metrics.jsonl"), ctx.config).show()